# 1. Data and features

**Question for the project:** what actually predicts a player's fantasy points *next week*?

This notebook loads every regular season nflverse has (1999-2025) of weekly player stats and Vegas lines
from [nflverse](https://github.com/nflverse/nflverse-data), defines the sample, and builds features that only
use information available **before** kickoff. Notebooks 2 and 3 focus on the modern game (2021-2025);
notebook 4 uses the full history. That last part matters: a feature computed with the
week it's predicting in it would make every model look better than it is.

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fantasy.data import ALL_SEASONS, SEASONS, load_player_weeks, load_games
from fantasy.features import FEATURES, TARGET, build_features, model_sample, mingus_projection
from fantasy import plots as P

P.use_style()
pd.set_option("display.precision", 3)

In [2]:
players = load_player_weeks(ALL_SEASONS)
games = load_games(ALL_SEASONS)
print(f"{len(players):,} player-games, {players.player_id.nunique():,} players, seasons {players.season.min()}-{players.season.max()}")
players[["player_display_name", "position", "season", "week", "team", "opponent_team",
         "targets", "carries", "receiving_yards", "rushing_yards", "fantasy_points_ppr"]].sample(5, random_state=1)

143,680 player-games, 3,622 players, seasons 1999-2025


,player_display_name,position,season,week,team,opponent_team,targets,carries,receiving_yards,rushing_yards,fantasy_points_ppr
89059,Mohamed Sanu,WR,2020,13,DET,CHI,2.0,0,36,0,5.60
86717,Bernard Pierce,RB,2014,6,BAL,TB,0.0,15,0,32,9.20
89957,Ryan Tannehill,QB,2013,7,MIA,BUF,0.0,3,0,5,14.26
98974,Sammy Watkins,WR,2017,16,LA,TEN,7.0,0,30,0,12.00
59825,Delanie Walker,TE,2010,7,SF,CAR,0.0,0,0,0,0.00


Two data quirks, handled in `fantasy/data.py`:
* **Relocated teams.** Player stats label every season with today's franchise code (LV, LAC, LA), while the
  schedule uses the city at the time (OAK, SD, STL). The schedule is mapped to franchise codes so every game finds its line.
* **Missing targets, 2003-2008.** nflverse has receptions but almost no targets for those seasons. They're set to
  missing rather than zero, so no feature quietly treats them as zero-target seasons.

In [3]:
print(players.groupby("season").targets.sum().loc[2001:2010].astype("Int64").to_dict())

{2001: 14903, 2002: 16127, 2003: 0, 2004: 0, 2005: 0, 2006: 0, 2007: 0, 2008: 0, 2009: 16343, 2010: 16607}


## Scoring check

nflverse ships PPR points already computed. A quick check against a hand calculation
(1 per 25 passing yards, 4 per passing TD, -2 per INT, 1 per 10 rushing/receiving yards, 6 per TD,
1 per reception, -2 per fumble lost, 2 per 2-point conversion) confirms we know what the target means.

In [4]:
hand = (players.passing_yards / 25 + 4 * players.passing_tds - 2 * players.passing_interceptions
        + (players.rushing_yards + players.receiving_yards) / 10
        + 6 * (players.rushing_tds + players.receiving_tds) + players.receptions
        - 2 * players.fumbles_lost + 2 * players.two_pt)
diff = (players.fantasy_points_ppr - hand).abs()
print(f"matches within 0.5 pts: {(diff < 0.5).mean():.1%} of player-games; largest gap {diff.max():.1f}")

matches within 0.5 pts: 99.6% of player-games; largest gap 12.0


## Vegas lines

Each game's over/under and spread turn into an **implied team total**: what the betting market expects
each team to score. Home implied = (total + spread) / 2.

In [5]:
g = games.dropna(subset=["implied_total", "points"])
print(f"implied total vs. actual points: r = {g.implied_total.corr(g.points):.2f}, "
      f"mean absolute miss {(g.implied_total - g.points).abs().mean():.1f} pts")
g[["season", "week", "team", "opponent_team", "total_line", "spread", "implied_total", "points"]].head()

implied total vs. actual points: r = 0.39, mean absolute miss 7.5 pts


,season,week,team,opponent_team,total_line,spread,implied_total,points
0,1999,1,ATL,MIN,49.0,-4.0,22.50,14.0
1,1999,1,CHI,KC,38.0,-3.0,17.50,20.0
2,1999,1,CLE,PIT,37.0,-6.0,15.50,0.0
3,1999,1,GB,LV,43.0,9.0,26.00,28.0
4,1999,1,IND,BUF,45.5,-3.0,21.25,31.0


## Features

For each player-game, `build_features` adds:

| Feature | Meaning (all from games *before* this week) |
|---|---|
| `last_game`, `avg_last3`, `avg_season` | PPR points last game, last 3 games, season to date |
| `prior_season_avg` | last season's PPR per game (4+ games) |
| `opps_per_game`, `targets_per_game`, `carries_per_game`, `target_share_avg` | volume |
| `tds_per_game` | touchdowns per game (rushing + receiving) |
| `def_factor` | PPR allowed per game by this week's opponent to the position, ÷ league average (1.0 = average) |
| `implied_total`, `implied_factor`, `spread`, `home` | Vegas: team total, total ÷ that week's average, expected margin, home game |

In [6]:
df = build_features(players, games)
sample = model_sample(df)
print(f"model sample, all seasons: {len(sample):,} player-games")
sample = sample[sample.season.isin(SEASONS)]
print(f"typical week: players score {sample[TARGET].sum() / sample.avg_season.sum():.1%} of their season average")
print(f"model sample, 2021-2025: {len(sample):,} player-games "
      f"(players with 3+ earlier games this season averaging 5+ PPR)")
sample.groupby("position").agg(rows=(TARGET, "size"), avg_ppr=(TARGET, "mean"), sd_ppr=(TARGET, "std"))

model sample, all seasons: 65,830 player-games
typical week: players score 97.5% of their season average
model sample, 2021-2025: 13,107 player-games (players with 3+ earlier games this season averaging 5+ PPR)


,rows,avg_ppr,sd_ppr
position,,,
QB,2043,15.888,8.311
RB,3474,11.480,8.376
TE,2070,9.090,6.982
WR,5520,10.548,8.134


## Leakage check

If the features are built right, `avg_season` for week *t* must equal the mean of the player's
actual points in weeks before *t*. Spot check one player, recomputed by hand:

In [7]:
name = "Ja'Marr Chase"
p = df[(df.player_display_name == name) & (df.season == 2024)][["week", TARGET, "last_game", "avg_last3", "avg_season"]]
p["check"] = p[TARGET].expanding().mean().shift(1)
assert np.allclose(p.avg_season.dropna(), p.check.dropna())
p.head(6)

,week,fantasy_points_ppr,last_game,avg_last3,avg_season,check
133448,1,12.2,NaN,NaN,NaN,NaN
133449,2,7.5,12.2,12.200,12.20,12.20
133450,3,29.8,7.5,9.850,9.85,9.85
133451,4,17.5,29.8,16.500,16.50,16.50
133452,5,41.3,17.5,18.267,16.75,16.75
133453,6,12.2,41.3,29.533,21.66,21.66


In [8]:
df.to_parquet("../data/features.parquet")
print("saved data/features.parquet")

saved data/features.parquet
